# Notebook 07 — Fine-Tuning AH-PC, Matching Sheibani's Code Exactly

This notebook reproduces Sheibani's fine-tuning code as literally as possible,
including a data-splitting detail that differs from every earlier fine-tuning
notebook in this series (`04_finetune.ipynb`, `04c`, `04d`, `04e`, `06`):

- **Earlier notebooks:** subject-dependent, leave-one-session-out 3-fold CV — each
  (subject, fold) trains and tests on that *subject's own* held-out session.
- **This notebook:** all 16 subjects' pairs are **pooled into one dataset**, and a
  single **global random 80/20 stratified split** (`test_size=0.2, stratify=y,
  random_state=42`) is computed **once** and reused, unchanged, for every one of
  the 48 (subject, fold) runs below. What varies per run is *only* which
  per-subject-per-fold pretrained checkpoint initialises the encoder — the
  training data, test data, and every hyperparameter stay fixed across all 48 runs.

**Architecture:** `CNNPredictiveCodingDE_Attn` (BandAttention + ChannelAttention +
CNN + Bottleneck) + `EEGEmotionClassifier` head
(`Flatten(4800) → Dropout(0.3) → Linear(4800,128) → ReLU → Linear(128,5)`) — no
256-d projector/embedding layer this time; the classifier reads the flattened
bottleneck feature map directly.

**Freezing:** only `encoder.encoder[0:8]` (the plain CNN Sequential — 2×
`Conv2d+BatchNorm2d+activation+MaxPool2d` blocks) has `requires_grad=False`.
`BandAttention`, `ChannelAttention`, and `bottleneck` all stay trainable. Note:
freezing here only stops gradient updates — the frozen CNN's `BatchNorm2d` layers
still update their running statistics during `.train()` forward passes, since
nothing forces them into `.eval()` (Sheibani's code doesn't either).

**Optimizer:** `Adam` (not `AdamW`), `lr=8e-5`, `weight_decay=1e-6`,
`StepLR(step_size=5, gamma=0.5)`, unweighted `CrossEntropyLoss`, batch size 64,
25 epochs. **No smoke test** — `RUN_FULL_TRAINING = True` by default, so running
this notebook launches all 48 runs immediately.

In [ ]:
import os
import copy
import pickle

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {DEVICE}")

DATA_DIR = "/workspaces/AH-PC/src/data/processed"
PER_SUBJECT_CKPT_DIR = "per_subject_k4"
FINETUNE_CKPT_DIR = "/workspaces/AH-PC/src/models/finetune_exact_sheibani"

EMOTIONS = ["Disgust", "Fear", "Sad", "Neutral", "Happy"]
NUM_SUBJECTS = 16
NUM_FOLDS = 3


def get_subject_fold_ckpt_path(subject_id, fold_idx):
    return os.path.join(
        PER_SUBJECT_CKPT_DIR, f"ahpc_subject{subject_id:02d}_fold{fold_idx + 1}_best.pt"
    )


## 1. Pooling All 16 Subjects and Building the Global 80/20 Split

Every subject's `subject_{sid:02d}_pairs.pkl` is loaded and concatenated into one
pool of segments. A single stratified 80/20 split is drawn from that pool **once**
— this exact split (as tensors) is what every one of the 48 runs below trains and
tests on.

In [ ]:
all_pairs = []
for subject_id in range(1, NUM_SUBJECTS + 1):
    path = os.path.join(DATA_DIR, f"subject_{subject_id:02d}_pairs.pkl")
    with open(path, "rb") as f:
        all_pairs.extend(pickle.load(f))

print(f"Pooled {len(all_pairs)} segments across {NUM_SUBJECTS} subjects.")

X_all = np.stack([np.asarray(p["anchor"], dtype=np.float32) for p in all_pairs])
y_all = np.array([p["label"] for p in all_pairs], dtype=np.int64)

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, stratify=y_all, random_state=42
)

print(f"Global train set: {X_train.shape[0]} segments")
print(f"Global test set:  {X_test.shape[0]} segments")
print("Train label distribution:", dict(zip(*np.unique(y_train, return_counts=True))))
print("Test label distribution: ", dict(zip(*np.unique(y_test, return_counts=True))))

train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
test_ds = TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test))
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

print("\nThis train_loader/test_loader pair is reused, unchanged, for all 48 "
      "(subject, fold) runs below.")


## 2. Architecture: `CNNPredictiveCodingDE_Attn` + `EEGEmotionClassifier`

`CNNPredictiveCodingDE_Attn` is the same attention + CNN + bottleneck encoder used
throughout this project (`SheibaniEncoder` in earlier notebooks, renamed here to
match Sheibani's own naming). `EEGEmotionClassifier` wraps it with the exact head
Sheibani used: `Flatten(4800) → Dropout(0.3) → Linear(4800,128) → ReLU →
Linear(128,5)` — no intermediate 256-d projector.

In [ ]:
class BandAttention(nn.Module):
    def __init__(self, num_bands):
        super().__init__()
        self.fc1 = nn.Linear(num_bands * 2, num_bands // 2)
        self.fc2 = nn.Linear(num_bands // 2, num_bands)

    def forward(self, x):  # x: (B, C, num_bands)
        avg_pool = x.mean(dim=1)
        max_pool = x.max(dim=1).values
        context  = torch.cat([avg_pool, max_pool], dim=-1)
        weights  = torch.sigmoid(self.fc2(F.relu(self.fc1(context))))
        return x * weights.unsqueeze(1)


class ChannelAttention(nn.Module):
    def __init__(self, num_channels):
        super().__init__()
        self.fc1 = nn.Linear(num_channels * 2, num_channels // 2)
        self.fc2 = nn.Linear(num_channels // 2, num_channels)

    def forward(self, x):  # x: (B, C, num_bands)
        avg_pool = x.mean(dim=2)
        max_pool = x.max(dim=2).values
        context  = torch.cat([avg_pool, max_pool], dim=-1)
        weights  = torch.sigmoid(self.fc2(F.relu(self.fc1(context))))
        return x * weights.unsqueeze(2)


class CNNPredictiveCodingDE_Attn(nn.Module):
    """Sheibani's attention + CNN encoder (identical to SheibaniEncoder in
    Notebooks 02/03/03b/03c/04c/04d/04e/06, renamed to match her own class name)."""

    OUT_CHANNELS = 64

    def __init__(self):
        super().__init__()
        self.band_attn = BandAttention(num_bands=5)
        self.channel_attn = ChannelAttention(num_channels=62)

        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.LeakyReLU(0.1),
            nn.MaxPool2d((2, 1)),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d((2, 1)),
        )

        self.bottleneck = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
        )

    def forward(self, x):  # x: (B, 62, 5)
        x = self.band_attn(x)
        x = self.channel_attn(x)
        x = x.unsqueeze(1)
        x = self.encoder(x)
        x = self.bottleneck(x)
        return x  # (B, 64, 15, 5)


BOTTLENECK_FLAT = 64 * 15 * 5  # 4800 — flattened (C, H, W) bottleneck


class EEGEmotionClassifier(nn.Module):
    """CNNPredictiveCodingDE_Attn + Sheibani's exact classification head:
    Flatten(4800) -> Dropout(0.3) -> Linear(4800,128) -> ReLU -> Linear(128,5).
    No projector / intermediate embedding layer."""

    def __init__(self, encoder, num_classes=5):
        super().__init__()
        self.encoder = encoder
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(BOTTLENECK_FLAT, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        feat = self.encoder(x)
        return self.classifier(feat)


print("CNNPredictiveCodingDE_Attn + EEGEmotionClassifier defined.")


## 3. Loading a Per-Subject-Per-Fold Checkpoint and Freezing the CNN

Only `sheibani_encoder.*` tensors are restored from each
`ahpc_subject{sid:02d}_fold{fold}_best.pt` checkpoint (produced by
`03c_pretrain_per_subject.ipynb`) — there is no `projector.*` in this
architecture, so those tensors in the checkpoint are simply unused.

Freezing: `for p in model.encoder.encoder[0:8].parameters(): p.requires_grad =
False` — this covers the entire plain CNN `Sequential` (all 8 of its submodules).
`band_attn`, `channel_attn`, and `bottleneck` are left untouched (trainable).

In [ ]:
def load_pretrained_encoder(ckpt_path):
    encoder = CNNPredictiveCodingDE_Attn()
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    full_sd = ckpt["model_state_dict"]
    enc_sd = {
        k[len("sheibani_encoder."):]: v
        for k, v in full_sd.items() if k.startswith("sheibani_encoder.")
    }
    encoder.load_state_dict(enc_sd)
    return encoder


def freeze_cnn_layers(model):
    """Freezes model.encoder.encoder[0:8] (the plain CNN Sequential) only.
    BandAttention, ChannelAttention, and bottleneck stay trainable."""
    for layer in model.encoder.encoder[0:8]:
        for p in layer.parameters():
            p.requires_grad = False


_ckpt_path = get_subject_fold_ckpt_path(1, 0)
_encoder = load_pretrained_encoder(_ckpt_path)
_model = EEGEmotionClassifier(_encoder)
freeze_cnn_layers(_model)
_trainable = sum(p.numel() for p in _model.parameters() if p.requires_grad)
_frozen = sum(p.numel() for p in _model.parameters() if not p.requires_grad)
print(f"Loaded from: {_ckpt_path}")
print(f"Frozen params (encoder.encoder[0:8], CNN only): {_frozen:,}")
print(f"Trainable params (band_attn + channel_attn + bottleneck + classifier): {_trainable:,}")
del _ckpt_path, _encoder, _model, _trainable, _frozen


## 4. Training: One Run Per (Subject, Fold), All on the Same Global Split

For each of the 48 `(subject, fold)` combinations: load that pair's own
pretrained checkpoint, build a fresh `EEGEmotionClassifier`, freeze the CNN, and
fine-tune on the **same global `train_loader`/`test_loader`** built in Section 1.

- **Optimizer:** `Adam`, `lr=8e-5`, `weight_decay=1e-6`
- **Scheduler:** `StepLR(step_size=5, gamma=0.5)`
- **Loss:** unweighted `CrossEntropyLoss`
- **Epochs:** 25 · **Batch size:** 64

Best-test-accuracy weights save to
`src/models/finetune_exact_sheibani/subject{sid:02d}_fold{fold}_best.pt`.

In [ ]:
def train_one_run(subject_id, fold_idx, epochs=25, lr=8e-5, weight_decay=1e-6,
                   step_size=5, gamma=0.5, device=DEVICE, verbose=False):
    ckpt_path = get_subject_fold_ckpt_path(subject_id, fold_idx)
    encoder = load_pretrained_encoder(ckpt_path)
    model = EEGEmotionClassifier(encoder).to(device)
    freeze_cnn_layers(model)

    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.Adam(trainable_params, lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=step_size, gamma=gamma)
    criterion = nn.CrossEntropyLoss()

    best_acc = -1.0
    best_state = None
    best_preds, best_labels = None, None

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        for anchor, label in train_loader:
            anchor, label = anchor.to(device), label.to(device)
            optimizer.zero_grad()
            logits = model(anchor)
            loss = criterion(logits, label)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * anchor.size(0)
        scheduler.step()
        avg_loss = running_loss / len(train_ds)

        model.eval()
        preds, labels = [], []
        with torch.no_grad():
            for anchor, label in test_loader:
                logits = model(anchor.to(device))
                preds.extend(logits.argmax(dim=1).cpu().numpy())
                labels.extend(label.numpy())
        acc = accuracy_score(labels, preds)

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())
            best_preds, best_labels = preds, labels

        if verbose:
            print(f"    Epoch {epoch:2d}/{epochs}  loss={avg_loss:.4f}  test_acc={acc:.4f}")

    os.makedirs(FINETUNE_CKPT_DIR, exist_ok=True)
    out_path = os.path.join(FINETUNE_CKPT_DIR, f"subject{subject_id:02d}_fold{fold_idx + 1}_best.pt")
    torch.save({"model_state_dict": best_state, "best_acc": best_acc}, out_path)

    return {
        "subject": subject_id,
        "fold": fold_idx + 1,
        "accuracy": best_acc,
        "y_true": np.array(best_labels),
        "y_pred": np.array(best_preds),
        "ckpt_path": out_path,
        "pretrained_ckpt_path": ckpt_path,
    }


print("train_one_run() defined.")


## 5. Full Experiment: All 16 Subjects × 3 Folds (No Smoke Test)

`RUN_FULL_TRAINING = True` by default — running this cell launches all 48 runs
(25 epochs each) immediately, all against the same global 80/20 split.

In [ ]:
RUN_FULL_TRAINING = True  # matches Sheibani's protocol: full run, no smoke test gate

fold_results = []  # one dict per (subject, fold), from train_one_run()

if RUN_FULL_TRAINING:
    for subject_id in range(1, NUM_SUBJECTS + 1):
        print(f"\nSubject {subject_id}")
        for fold_idx in range(NUM_FOLDS):
            res = train_one_run(subject_id, fold_idx, epochs=25, verbose=False)
            fold_results.append(res)
            print(f"  Fold {fold_idx + 1}/3 -> best test_acc = {res['accuracy']:.4f}")
    print("\nTraining complete for all subjects and folds.")
else:
    print("RUN_FULL_TRAINING is False — skipping the full 16-subject x 3-fold run.")


## 6. Evaluation

- **Per-subject mean accuracy** across its 3 folds (± std) — recall that all 3
  folds for a subject share the *same* global test set; only the initial
  checkpoint differs per fold.
- **Overall mean ± std across all 16 subjects**
- An **aggregated classification report** and **normalized confusion matrix**

In [ ]:
from collections import defaultdict


def plot_confusion_matrix(cm, class_names=EMOTIONS, title="Confusion Matrix (Normalized)"):
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(class_names)))
    ax.set_xticklabels(class_names, rotation=45, ha="right")
    ax.set_yticks(range(len(class_names)))
    ax.set_yticklabels(class_names)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(title)
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            ax.text(j, i, f"{cm_norm[i, j]:.2f}", ha="center", va="center",
                    color="white" if cm_norm[i, j] > 0.5 else "black")
    fig.colorbar(im, ax=ax, label="Fraction of true class")
    plt.tight_layout()
    plt.show()


overall_mean = overall_std = None

if not fold_results:
    print("No results yet — set RUN_FULL_TRAINING=True in Section 5 and re-run.")
else:
    acc_by_subject = defaultdict(list)
    for r in fold_results:
        acc_by_subject[r["subject"]].append(r["accuracy"])

    print("Per-subject mean accuracy across 3 folds:")
    subject_means = []
    for sid in sorted(acc_by_subject):
        accs = acc_by_subject[sid]
        m, s = np.mean(accs), np.std(accs)
        subject_means.append(m)
        fold_str = ", ".join(f"{a * 100:.2f}%" for a in accs)
        print(f"  Subject {sid:2d}: {m * 100:5.2f}% ± {s * 100:5.2f}%  (folds: {fold_str})")

    overall_mean = float(np.mean(subject_means) * 100)
    overall_std = float(np.std(subject_means) * 100)
    print(f"\nOverall mean accuracy across all {len(subject_means)} subjects: "
          f"{overall_mean:.2f}% ± {overall_std:.2f}%")

    all_y_true = np.concatenate([r["y_true"] for r in fold_results])
    all_y_pred = np.concatenate([r["y_pred"] for r in fold_results])

    print("\nAggregated classification report (all 48 runs pooled):")
    print(classification_report(all_y_true, all_y_pred, target_names=EMOTIONS, digits=3))

    agg_cm = confusion_matrix(all_y_true, all_y_pred, labels=[0, 1, 2, 3, 4])
    plot_confusion_matrix(agg_cm, title="Aggregated Confusion Matrix (Normalized) — Exact Sheibani Protocol")


## 7. Comparison Against the PC-SSL Baseline

In [ ]:
PC_SSL_BASELINE_MEAN = 78.36
PC_SSL_BASELINE_STD = 16.37
AHPC_K1_MEAN = 91.06
AHPC_K1_STD = 2.29
k4_label = "AH-PC k∈{1,2,4,8}:"

print(f"{'PC-SSL baseline:':20s} {PC_SSL_BASELINE_MEAN:.2f}% ± {PC_SSL_BASELINE_STD:.2f}%")
print(f"{'AH-PC k=1:':20s} {AHPC_K1_MEAN:.2f}% ± {AHPC_K1_STD:.2f}%")

if overall_mean is None:
    print(f"{k4_label:20s} XX.XX% ± XX.XX%")
    print("\nRun Section 5 (RUN_FULL_TRAINING=True) to fill this in.")
else:
    print(f"{k4_label:20s} {overall_mean:.2f}% ± {overall_std:.2f}%")
    print(f"\n{'vs. PC-SSL baseline:':20s} {overall_mean - PC_SSL_BASELINE_MEAN:+.2f} pts")
    print(f"{'vs. AH-PC k=1:':20s} {overall_mean - AHPC_K1_MEAN:+.2f} pts")


---
## Summary

| Step | Status |
|------|--------|
| Pooled all 16 subjects, drew one global 80/20 stratified split | done |
| `CNNPredictiveCodingDE_Attn` + `EEGEmotionClassifier` defined (no projector) | done |
| CNN (`encoder.encoder[0:8]`) frozen; attention + bottleneck + head trainable | done |
| Full protocol: 48 runs, each from its own per-subject-per-fold checkpoint, same global split | depends on run |
| Comparison vs. PC-SSL baseline (78.36% ± 16.37%) and AH-PC k=1 (91.06% ± 2.29%) | depends on run |